# 3단계. 머신러닝 분석·학습·평가

1단계의 17개 데이터테이블을 이용해 다음 순서로 진행합니다.

1. 가설과 시나리오 정의
2. 데이터 전처리
3. 한글 컬럼 확인용 데이터 저장
4. EDA
5. 원본 통화 단위 학습·검증·테스트 분리
6. 여러 알고리즘 학습·비교
7. 시나리오별 최종 모델 선정과 최종 테스트

모델은 검증 데이터로 비교하며, 최종 테스트 데이터는 선정된 모델에 한 번만 사용합니다.

In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow scikit-learn seaborn matplotlib koreanize-matplotlib joblib openpyxl

In [ ]:
# 1. 라이브러리와 Google Drive
from google.colab import drive
from pathlib import Path
from IPython.display import display
import hashlib, json, re, unicodedata, warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

from sklearn.base import clone
from sklearn.cluster import AgglomerativeClustering, KMeans, MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, average_precision_score,
    classification_report, confusion_matrix,
    precision_recall_fscore_support, silhouette_score
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings('ignore')
drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 분석 설정

In [ ]:
# 2. 경로와 설정값
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
DATASET_ROOT = PROJECT_ROOT / '구축 데이터셋_v3'
STANDARD_ROOT = DATASET_ROOT / '01_standard_tables'
ML_ROOT = DATASET_ROOT / '02_ml_tables'
OUTPUT_ROOT = PROJECT_ROOT / '머신러닝_분석결과_v3'

KOREAN_ROOT = OUTPUT_ROOT / '00_한글확인용'
EDA_ROOT = OUTPUT_ROOT / '01_EDA'
SPLIT_ROOT = OUTPUT_ROOT / '02_데이터분리'
MODEL_ROOT = OUTPUT_ROOT / '03_모델'
PRED_ROOT = OUTPUT_ROOT / '04_예측결과'
REPORT_ROOT = OUTPUT_ROOT / '05_보고서'
for folder in [KOREAN_ROOT, EDA_ROOT, SPLIT_ROOT, MODEL_ROOT, PRED_ROOT, REPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
TEST_RATIO = 0.20
DEV_RATIO = 0.20
MAX_FEATURES = 50000
MIN_TEXT_LENGTH = 10

assert ML_ROOT.exists(), f'1단계 결과 경로를 확인하세요: {ML_ROOT}'
print('입력:', DATASET_ROOT)
print('출력:', OUTPUT_ROOT)

## 2. 가설 및 분석 시나리오

In [ ]:
# 3. 이번 노트북에서 검증할 핵심 가설
scenario_df = pd.DataFrame([
    ['H1', '정상 금융상담과 보이스피싱은 텍스트로 구분할 수 있다',
     '정상상담 vs 보이스피싱', 'Recall·F1·PR-AUC'],
    ['H3', '전체 대화와 임의 구간에서도 보이스피싱을 탐지할 수 있다',
     '전체·부분 구간 탐지', '구간별 Recall·F1·PR-AUC'],
    ['H5', '대출사기형과 수사기관형은 언어적 특징이 다르다',
     '보이스피싱 유형 분류', 'Macro F1·혼동행렬'],
    ['E1', '기존 분류보다 세분된 유사 사건 유형 후보가 존재한다',
     '유사 사건 군집화', 'Silhouette·군집 안정성'],
], columns=['가설ID', '가설', '분석시나리오', '평가지표'])
display(scenario_df)
scenario_df.to_csv(REPORT_ROOT / '분석_시나리오.csv', index=False, encoding='utf-8-sig')

print('H2·H4·H6·H7·H8은 사칭·행동·전략 SILVER 변수를 이용한 EDA 보조가설로 해석합니다.')
print('실제 피해 여부 정답이 없으므로 피해 발생 확률은 예측하지 않습니다.')

## 3. 데이터 불러오기 및 전처리

In [ ]:
# 4. 1단계 데이터 불러오기
def read_table(folder, name):
    parquet_path = folder / f'{name}.parquet'
    csv_path = folder / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    assert csv_path.exists(), f'테이블을 찾지 못했습니다: {name}'
    return pd.read_csv(csv_path, encoding='utf-8-sig')

detection_df = read_table(ML_ROOT, 'fraud_detection_ml')
type_df = read_table(ML_ROOT, 'fraud_type_ml')
segment_df = read_table(ML_ROOT, 'segment_detection_ml')
cluster_df = read_table(ML_ROOT, 'case_clustering_ml')
amount_df = read_table(STANDARD_ROOT, 'vp_amount_events')
case_df = read_table(STANDARD_ROOT, 'vp_cases')

summary_df = pd.DataFrame([
    ['정상·사기', len(detection_df), len(detection_df.columns)],
    ['사기유형', len(type_df), len(type_df.columns)],
    ['전체·부분구간', len(segment_df), len(segment_df.columns)],
    ['사건군집', len(cluster_df), len(cluster_df.columns)],
    ['금액이벤트', len(amount_df), len(amount_df.columns)],
], columns=['데이터', '행수', '컬럼수'])
display(summary_df)
assert len(detection_df) and len(type_df) and len(segment_df) and len(cluster_df)

In [ ]:
# 5. 모델 입력 텍스트 정리
# 원문 컬럼은 보존하고 clean_text만 모델에 사용합니다.
def clean_text(text):
    text = unicodedata.normalize('NFKC', str(text or '')).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'(?m)^\s*(tx|rx|화자\s*\d*|범인|피해자)\s*[:：]?\s*', ' ', text)
    text = re.sub(r'[*#xX]{2,}', ' 마스킹 ', text)
    text = re.sub(r'\b\d{2,}\b', ' 숫자 ', text)
    return re.sub(r'\s+', ' ', text).strip()

def prepare(df, source_text):
    result = df.copy()
    result['clean_text'] = result[source_text].fillna('').map(clean_text)
    result = result[result['clean_text'].str.len() >= MIN_TEXT_LENGTH].copy()
    result['text_hash'] = result['clean_text'].map(
        lambda x: hashlib.sha256(x.encode('utf-8')).hexdigest()
    )
    return result.reset_index(drop=True)

def remove_conflicts_and_duplicates(df, label):
    conflict_hashes = df.groupby('text_hash')[label].nunique()
    conflict_hashes = set(conflict_hashes[conflict_hashes > 1].index)
    conflict_count = int(df['text_hash'].isin(conflict_hashes).sum())
    result = df[~df['text_hash'].isin(conflict_hashes)].copy()
    duplicate_count = int(result.duplicated([label, 'text_hash']).sum())
    result = result.drop_duplicates([label, 'text_hash']).reset_index(drop=True)
    return result, conflict_count, duplicate_count

detection_df = prepare(detection_df, 'model_input_text')
type_df = prepare(type_df, 'model_input_text')
segment_df = prepare(segment_df, 'window_text')
cluster_df = prepare(cluster_df, 'model_input_text')

detection_df, d_conflicts, d_duplicates = remove_conflicts_and_duplicates(detection_df, 'fraud_label')
type_df, t_conflicts, t_duplicates = remove_conflicts_and_duplicates(type_df, 'supervised_target')

if 'amount_krw' in amount_df:
    amount_df['amount_10k_krw'] = pd.to_numeric(amount_df['amount_krw'], errors='coerce') / 10000
    amount_df['log_amount_10k_krw'] = np.log1p(amount_df['amount_10k_krw'])

display(pd.DataFrame([
    ['정상·사기', d_conflicts, d_duplicates, len(detection_df)],
    ['사기유형', t_conflicts, t_duplicates, len(type_df)],
], columns=['데이터', '라벨충돌제외', '정확중복제외', '최종행수']))
print('모델 입력은 clean_text 하나이며 출처·경로·파일명은 사용하지 않습니다.')

## 4. 사람이 확인할 수 있는 한글 컬럼 데이터

In [ ]:
# 6. 한글 컬럼 CSV 저장
column_ko = {
    'conversation_id':'대화ID', 'case_id':'사건ID', 'file_id':'원본파일ID',
    'group_id':'분리그룹ID', 'fraud_label':'사기여부',
    'supervised_target':'보이스피싱유형', 'source_group':'데이터출처',
    'sample_scope':'표본범위', 'window_position':'구간위치',
    'window_start_turn':'구간시작발화', 'window_end_turn':'구간종료발화',
    'window_size':'구간발화수', 'model_input_text':'모델입력원문',
    'window_text':'구간원문', 'clean_text':'정제텍스트',
    'financial_topic':'금융상담주제', 'quality_flag':'품질상태',
    'original_split':'원본데이터구분', 'label_source':'라벨출처',
    'source_category':'원본사기분류',
    'primary_impersonation_group':'주요사칭대분류',
    'primary_impersonation_subtype':'주요사칭세부유형',
    'primary_requested_action':'주요요구행동',
    'amount_krw':'금액_원', 'amount_10k_krw':'금액_만원',
    'amount_status':'금액상태', 'amount_direction':'금액방향',
    'amount_purpose':'금액용도', 'amount_direction_evidence':'금액방향_근거문장',
    'amount_direction_confidence':'금액방향_신뢰도', 'amount_text':'금액원문',
    'evidence_text':'근거발화', 'evidence_role':'근거화자역할',
}
def save_korean(df, filename):
    result = df.rename(columns=column_ko)
    result.to_csv(KOREAN_ROOT / filename, index=False, encoding='utf-8-sig')
    return result

detection_ko = save_korean(detection_df, '정상상담_보이스피싱_분류데이터.csv')
save_korean(type_df, '보이스피싱_유형분류데이터.csv')
save_korean(segment_df, '전체_부분구간_탐지데이터.csv')
save_korean(cluster_df, '유사사건_군집데이터.csv')
save_korean(amount_df, '금액이벤트_원_만원.csv')

display(detection_ko.head(3))
print('한글 확인용 CSV 5개 저장:', KOREAN_ROOT)

## 5. EDA

In [ ]:
# 7. 분포와 텍스트 길이 확인
detection_df['text_length'] = detection_df['clean_text'].str.len()
type_df['text_length'] = type_df['clean_text'].str.len()

eda_tables = {
    '정상사기_분포': detection_df['fraud_label'].value_counts().rename_axis('구분').reset_index(name='건수'),
    '사기유형_분포': type_df['supervised_target'].value_counts().rename_axis('유형').reset_index(name='건수'),
    '구간위치_분포': segment_df.groupby(
        ['sample_scope','window_position','fraud_label']
    ).size().reset_index(name='건수'),
}
if 'amount_status' in amount_df:
    eda_tables['금액상태_분포'] = amount_df['amount_status'].value_counts(
        dropna=False
    ).rename_axis('상태').reset_index(name='건수')

for name, frame in eda_tables.items():
    display(frame)
    frame.to_csv(EDA_ROOT / f'{name}.csv', index=False, encoding='utf-8-sig')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(data=detection_df, x='fraud_label', ax=axes[0])
axes[0].set_title('정상상담과 보이스피싱 분포')
axes[0].tick_params(axis='x', rotation=15)
sns.boxplot(data=detection_df, x='fraud_label', y='text_length', showfliers=False, ax=axes[1])
axes[1].set_title('정제 텍스트 길이')
axes[1].tick_params(axis='x', rotation=15)
sns.countplot(data=type_df, x='supervised_target', ax=axes[2])
axes[2].set_title('보이스피싱 유형 분포')
axes[2].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(EDA_ROOT / '기본_EDA.png', dpi=160, bbox_inches='tight')
plt.show()

if 'amount_10k_krw' in amount_df:
    upper = amount_df['amount_10k_krw'].quantile(.99)
    plot_amount = amount_df[amount_df['amount_10k_krw'].between(0, upper)]
    plt.figure(figsize=(10,5))
    sns.histplot(data=plot_amount, x='amount_10k_krw', hue='amount_status', bins=30)
    plt.xlabel('금액(만원)')
    plt.title('금액 상태별 분포: 상위 1% 이상치 제외')
    plt.tight_layout()
    plt.savefig(EDA_ROOT / '금액분포_만원.png', dpi=160, bbox_inches='tight')
    plt.show()

## 6. 학습·검증·최종 테스트 분리

정상 상담의 공식 TRAIN/VALIDATION을 보존합니다. 보이스피싱은 원본 file_id에 해당하는 group_id로 분리합니다. 같은 원본 통화의 사건·구간이 서로 다른 세트에 들어가면 실행을 중단합니다.

In [ ]:
# 8. 그룹 분리 공통 함수
def group_split_map(df, label_col, group_col='group_id'):
    group_labels = df.groupby(group_col)[label_col].agg(
        lambda s: s.mode().iloc[0]
    ).reset_index()
    mixed = df.groupby(group_col)[label_col].nunique().max()
    assert mixed == 1, '한 그룹에 서로 다른 정답이 섞여 있습니다.'
    train_dev, test = train_test_split(
        group_labels, test_size=TEST_RATIO, random_state=SEED,
        stratify=group_labels[label_col]
    )
    train, dev = train_test_split(
        train_dev, test_size=DEV_RATIO, random_state=SEED,
        stratify=train_dev[label_col]
    )
    result = {x:'TRAIN' for x in train[group_col]}
    result.update({x:'DEV' for x in dev[group_col]})
    result.update({x:'TEST' for x in test[group_col]})
    return result

def check_split(df, label_col):
    assert df.groupby('group_id')['ml_split'].nunique().max() == 1, '원본 통화 누출 발생'
    assert {'TRAIN','DEV','TEST'}.issubset(set(df['ml_split']))
    for split_name in ['TRAIN','DEV','TEST']:
        assert df.loc[df['ml_split'].eq(split_name), label_col].nunique() >= 2, (
            f'{split_name}에 클래스가 하나뿐입니다.'
        )
    return df.groupby(['ml_split',label_col]).size().reset_index(name='건수')

In [ ]:
# 9. 정상 vs 사기 분리
det = detection_df.copy()
det['ml_split'] = ''
normal = det['fraud_label'].eq('LEGITIMATE_FINANCIAL_CALL')
fraud = det['fraud_label'].eq('VOICE_PHISHING')
official = det['original_split'].fillna('').astype(str).str.upper()

normal_train_ids = det.loc[normal & official.eq('TRAIN'), 'group_id'].drop_duplicates()
normal_test_ids = det.loc[normal & official.eq('VALIDATION'), 'group_id'].drop_duplicates()
assert len(normal_train_ids) and len(normal_test_ids), '정상 상담 TRAIN/VALIDATION이 모두 필요합니다.'
normal_train_ids, normal_dev_ids = train_test_split(
    normal_train_ids, test_size=DEV_RATIO, random_state=SEED
)
det.loc[normal & det['group_id'].isin(normal_train_ids), 'ml_split'] = 'TRAIN'
det.loc[normal & det['group_id'].isin(normal_dev_ids), 'ml_split'] = 'DEV'
det.loc[normal & det['group_id'].isin(normal_test_ids), 'ml_split'] = 'TEST'

fraud_ids = det.loc[fraud, 'group_id'].drop_duplicates()
fraud_train_dev_ids, fraud_test_ids = train_test_split(
    fraud_ids, test_size=TEST_RATIO, random_state=SEED
)
fraud_train_ids, fraud_dev_ids = train_test_split(
    fraud_train_dev_ids, test_size=DEV_RATIO, random_state=SEED
)
det.loc[fraud & det['group_id'].isin(fraud_train_ids), 'ml_split'] = 'TRAIN'
det.loc[fraud & det['group_id'].isin(fraud_dev_ids), 'ml_split'] = 'DEV'
det.loc[fraud & det['group_id'].isin(fraud_test_ids), 'ml_split'] = 'TEST'
det = det[det['ml_split'].ne('')].copy()

display(check_split(det, 'fraud_label'))
det.to_csv(SPLIT_ROOT / '정상사기_데이터분리.csv', index=False, encoding='utf-8-sig')

# 유형 분리
fraud_type = type_df.rename(columns={'file_id':'group_id'}).copy()
fraud_type['ml_split'] = fraud_type['group_id'].map(
    group_split_map(fraud_type, 'supervised_target')
)
display(check_split(fraud_type, 'supervised_target'))
fraud_type.to_csv(SPLIT_ROOT / '사기유형_데이터분리.csv', index=False, encoding='utf-8-sig')

## 7. 여러 머신러닝 알고리즘 비교

In [ ]:
# 10. 모델 후보와 평가 함수
def word_vec():
    return TfidfVectorizer(
        ngram_range=(1,2), min_df=2, max_df=.98,
        sublinear_tf=True, max_features=MAX_FEATURES
    )
def char_vec():
    return TfidfVectorizer(
        analyzer='char_wb', ngram_range=(3,5), min_df=2,
        sublinear_tf=True, max_features=MAX_FEATURES
    )
def binary_candidates():
    return {
        'dummy': Pipeline([('tfidf',word_vec()),('model',DummyClassifier(strategy='prior'))]),
        'word_logistic': Pipeline([('tfidf',word_vec()),('model',LogisticRegression(
            max_iter=2000,class_weight='balanced',random_state=SEED))]),
        'char_logistic': Pipeline([('tfidf',char_vec()),('model',LogisticRegression(
            max_iter=2000,class_weight='balanced',random_state=SEED))]),
        'char_linear_svm': Pipeline([('tfidf',char_vec()),('model',LinearSVC(
            class_weight='balanced',random_state=SEED))]),
        'char_sgd': Pipeline([('tfidf',char_vec()),('model',SGDClassifier(
            loss='log_loss',class_weight='balanced',max_iter=2000,random_state=SEED))]),
        'word_complement_nb': Pipeline([('tfidf',word_vec()),('model',ComplementNB())]),
    }
def multiclass_candidates():
    result = binary_candidates()
    result.pop('dummy')
    return result

def positive_score(model, x, positive):
    index = list(model.classes_).index(positive)
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(x)[:,index]
    score = model.decision_function(x)
    if np.ndim(score) == 1:
        return score if index == 1 else -score
    return score[:,index]

def binary_scores(y_true, y_pred, score, positive):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='binary', pos_label=positive, zero_division=0
    )
    y_binary = (np.asarray(y_true) == positive).astype(int)
    return {
        'accuracy':accuracy_score(y_true,y_pred),
        'precision':precision, 'recall':recall, 'f1':f1,
        'pr_auc':average_precision_score(y_binary,score)
    }

def compare_binary(train, dev, label, positive):
    rows = []
    for name, model in binary_candidates().items():
        model.fit(train['clean_text'], train[label])
        pred = model.predict(dev['clean_text'])
        score = positive_score(model, dev['clean_text'], positive)
        rows.append({'모델':name, **binary_scores(dev[label],pred,score,positive)})
        print(name, '완료')
    return pd.DataFrame(rows).sort_values(['pr_auc','f1'],ascending=False)

def compare_multiclass(train, dev, label):
    rows = []
    for name, model in multiclass_candidates().items():
        model.fit(train['clean_text'], train[label])
        pred = model.predict(dev['clean_text'])
        p,r,f1,_ = precision_recall_fscore_support(
            dev[label],pred,average='macro',zero_division=0
        )
        rows.append({'모델':name,'accuracy':accuracy_score(dev[label],pred),
                     'macro_precision':p,'macro_recall':r,'macro_f1':f1})
        print(name, '완료')
    return pd.DataFrame(rows).sort_values('macro_f1',ascending=False)

### 7-1. 정상 금융상담 vs 보이스피싱

In [ ]:
# 11. 검증 데이터로 모델 선정 후 최종 테스트 1회
train = det[det['ml_split'].eq('TRAIN')]
dev = det[det['ml_split'].eq('DEV')]
test = det[det['ml_split'].eq('TEST')]

detection_compare = compare_binary(train,dev,'fraud_label','VOICE_PHISHING')
display(detection_compare)
best_detection_name = detection_compare.iloc[0]['모델']

best_detection = clone(binary_candidates()[best_detection_name])
train_dev = det[det['ml_split'].isin(['TRAIN','DEV'])]
best_detection.fit(train_dev['clean_text'],train_dev['fraud_label'])
det_pred = best_detection.predict(test['clean_text'])
det_score = positive_score(best_detection,test['clean_text'],'VOICE_PHISHING')
detection_test = binary_scores(
    test['fraud_label'],det_pred,det_score,'VOICE_PHISHING'
)
display(pd.DataFrame([detection_test]))
print(classification_report(test['fraud_label'],det_pred,zero_division=0))

det_result = test[['conversation_id','group_id','fraud_label','clean_text']].copy()
det_result['예측'] = det_pred
det_result['보이스피싱점수'] = det_score
det_result['정답여부'] = det_result['fraud_label'].eq(det_result['예측'])
det_result.to_csv(PRED_ROOT/'정상사기_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
detection_compare.to_csv(REPORT_ROOT/'정상사기_모델비교.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_detection,MODEL_ROOT/'fraud_detection_best_model.joblib')

labels = ['LEGITIMATE_FINANCIAL_CALL','VOICE_PHISHING']
cm = confusion_matrix(test['fraud_label'],det_pred,labels=labels)
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',
            xticklabels=['정상','보이스피싱'],yticklabels=['정상','보이스피싱'])
plt.xlabel('예측'); plt.ylabel('실제'); plt.title('정상 vs 보이스피싱')
plt.tight_layout(); plt.savefig(REPORT_ROOT/'정상사기_혼동행렬.png',dpi=160); plt.show()

### 7-2. 대출사기형 vs 수사기관 사칭형

In [ ]:
# 12. 유형 분류 모델 비교·선정·최종 테스트
train = fraud_type[fraud_type['ml_split'].eq('TRAIN')]
dev = fraud_type[fraud_type['ml_split'].eq('DEV')]
test_type = fraud_type[fraud_type['ml_split'].eq('TEST')]

type_compare = compare_multiclass(train,dev,'supervised_target')
display(type_compare)
best_type_name = type_compare.iloc[0]['모델']

best_type = clone(multiclass_candidates()[best_type_name])
train_dev = fraud_type[fraud_type['ml_split'].isin(['TRAIN','DEV'])]
best_type.fit(train_dev['clean_text'],train_dev['supervised_target'])
type_pred = best_type.predict(test_type['clean_text'])
p,r,type_f1,_ = precision_recall_fscore_support(
    test_type['supervised_target'],type_pred,average='macro',zero_division=0
)
type_test = {'accuracy':accuracy_score(test_type['supervised_target'],type_pred),
             'macro_precision':p,'macro_recall':r,'macro_f1':type_f1}
display(pd.DataFrame([type_test]))
print(classification_report(test_type['supervised_target'],type_pred,zero_division=0))

type_result = test_type[['case_id','group_id','supervised_target','clean_text']].copy()
type_result['예측'] = type_pred
type_result['정답여부'] = type_result['supervised_target'].eq(type_result['예측'])
type_result.to_csv(PRED_ROOT/'사기유형_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
type_compare.to_csv(REPORT_ROOT/'사기유형_모델비교.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_type,MODEL_ROOT/'fraud_type_best_model.joblib')

### 7-3. 전체 대화 및 임의 구간 탐지

In [ ]:
# 13. 같은 원본 통화 분할을 구간 데이터에도 적용
split_lookup = det[['group_id','ml_split']].drop_duplicates()
segment = segment_df.merge(split_lookup,on='group_id',how='inner')
display(check_split(segment,'fraud_label'))

train = segment[segment['ml_split'].eq('TRAIN')]
dev = segment[segment['ml_split'].eq('DEV')]
test_segment = segment[segment['ml_split'].eq('TEST')]

segment_compare = compare_binary(train,dev,'fraud_label','VOICE_PHISHING')
display(segment_compare)
best_segment_name = segment_compare.iloc[0]['모델']

best_segment = clone(binary_candidates()[best_segment_name])
train_dev = segment[segment['ml_split'].isin(['TRAIN','DEV'])]
best_segment.fit(train_dev['clean_text'],train_dev['fraud_label'])
segment_pred = best_segment.predict(test_segment['clean_text'])
segment_score = positive_score(best_segment,test_segment['clean_text'],'VOICE_PHISHING')

segment_result = test_segment.copy()
segment_result['예측'] = segment_pred
segment_result['보이스피싱점수'] = segment_score
segment_result['정답여부'] = segment_result['fraud_label'].eq(segment_result['예측'])

position_rows = []
for (scope,position), group in segment_result.groupby(['sample_scope','window_position']):
    if group['fraud_label'].nunique() < 2:
        continue
    result = binary_scores(
        group['fraud_label'],group['예측'],group['보이스피싱점수'],'VOICE_PHISHING'
    )
    position_rows.append({'표본범위':scope,'구간위치':position,'건수':len(group),**result})
position_df = pd.DataFrame(position_rows)
display(position_df)

segment_compare.to_csv(REPORT_ROOT/'구간탐지_모델비교.csv',index=False,encoding='utf-8-sig')
position_df.to_csv(REPORT_ROOT/'전체부분구간_위치별성능.csv',index=False,encoding='utf-8-sig')
segment_result.to_csv(PRED_ROOT/'전체부분구간_최종테스트_예측.csv',index=False,encoding='utf-8-sig')
joblib.dump(best_segment,MODEL_ROOT/'segment_detection_best_model.joblib')

### 7-4. 유사 사건 군집화

In [ ]:
# 14. K-means·MiniBatch K-means·계층 군집 비교
vectorizer = TfidfVectorizer(
    ngram_range=(1,2),min_df=2,max_df=.95,
    max_features=MAX_FEATURES,sublinear_tf=True
)
matrix = vectorizer.fit_transform(cluster_df['clean_text'])
n_components = max(2,min(50,matrix.shape[0]-1,matrix.shape[1]-1))
svd = TruncatedSVD(n_components=n_components,random_state=SEED)
features = svd.fit_transform(matrix)

rows, outputs = [], {}
for k in range(2,min(8,len(cluster_df)-1)+1):
    candidates = {
        'kmeans':KMeans(n_clusters=k,n_init=20,random_state=SEED),
        'minibatch_kmeans':MiniBatchKMeans(
            n_clusters=k,n_init=20,batch_size=256,random_state=SEED),
        'agglomerative':AgglomerativeClustering(n_clusters=k),
    }
    for name, model in candidates.items():
        labels = model.fit_predict(features)
        silhouette = silhouette_score(features,labels)
        stability = np.nan
        if name != 'agglomerative':
            other = clone(model).set_params(random_state=SEED+1)
            stability = adjusted_rand_score(labels,other.fit_predict(features))
        rows.append({'알고리즘':name,'군집수':k,
                     'silhouette':silhouette,'seed_stability_ari':stability})
        outputs[(name,k)] = (model,labels)

cluster_compare = pd.DataFrame(rows).sort_values(
    ['silhouette','seed_stability_ari'],ascending=False
)
display(cluster_compare.head(12))
best_cluster_row = cluster_compare.iloc[0]
best_cluster_key = (best_cluster_row['알고리즘'],int(best_cluster_row['군집수']))
best_cluster_model,cluster_labels = outputs[best_cluster_key]

cluster_result = cluster_df.copy()
cluster_result['군집ID'] = cluster_labels
cluster_result.to_csv(PRED_ROOT/'유사사건_군집결과.csv',index=False,encoding='utf-8-sig')
cluster_compare.to_csv(REPORT_ROOT/'군집모델_비교.csv',index=False,encoding='utf-8-sig')
joblib.dump({'vectorizer':vectorizer,'svd':svd,'model':best_cluster_model,
             'algorithm':best_cluster_key[0],'cluster_count':best_cluster_key[1]},
            MODEL_ROOT/'case_clustering_best_model.joblib')

terms = np.array(vectorizer.get_feature_names_out())
representatives = []
for cluster_id, group in cluster_result.groupby('군집ID'):
    indices = group.index.to_numpy()
    mean_tfidf = np.asarray(matrix[indices].mean(axis=0)).ravel()
    top_terms = terms[mean_tfidf.argsort()[-12:][::-1]]
    centroid = features[indices].mean(axis=0)
    local = np.linalg.norm(features[indices]-centroid,axis=1).argmin()
    item = group.iloc[local]
    representatives.append({
        '군집ID':cluster_id,'사건수':len(group),'대표단어':' | '.join(top_terms),
        '대표사건ID':item['case_id'],'대표문장':item['model_input_text'][:500]
    })
representative_df = pd.DataFrame(representatives)
display(representative_df)
representative_df.to_csv(REPORT_ROOT/'군집별_대표단어_대표사건.csv',index=False,encoding='utf-8-sig')

## 8. 시나리오별 최종 모델과 보고서

In [ ]:
# 15. 최종 모델 선정 결과
best_models = pd.DataFrame([
    ['정상상담 vs 보이스피싱',best_detection_name,'검증 PR-AUC 우선·F1 보조',
     'PR-AUC',detection_test['pr_auc']],
    ['보이스피싱 유형 분류',best_type_name,'검증 Macro F1',
     'Macro F1',type_test['macro_f1']],
    ['전체·부분 구간 탐지',best_segment_name,'검증 PR-AUC 우선·F1 보조',
     '구간별 평균 PR-AUC',position_df['pr_auc'].mean() if len(position_df) else np.nan],
    ['유사 사건 군집화',f'{best_cluster_key[0]} (k={best_cluster_key[1]})',
     'Silhouette·시드 안정성','Silhouette',best_cluster_row['silhouette']],
],columns=['시나리오','선정모델','선정기준','최종지표','최종점수'])
display(best_models)
best_models.to_csv(REPORT_ROOT/'시나리오별_최종모델.csv',index=False,encoding='utf-8-sig')

quality_path = PROJECT_ROOT/'데이터셋 품질테스트_v2'/'quality_scoring'/'quality_scoring_summary.json'
quality_note = '2-2단계 품질점수 결과 없음'
if quality_path.exists():
    q = json.loads(quality_path.read_text(encoding='utf-8'))
    quality_note = (
        f"검수 {q.get('completed_rows',0)}건, "
        f"전사 정확도 {q.get('transcription_accuracy',np.nan)*100:.2f}점, "
        f"역할 Macro F1 {q.get('role_macro_f1',np.nan):.4f}"
    )

report = [
    '# 3단계 머신러닝 분석 결과','',
    '## Summary','',
    f'- 정상·사기 최종 모델: {best_detection_name}',
    f"- PR-AUC: {detection_test['pr_auc']:.4f}",
    f"- 보이스피싱 Recall: {detection_test['recall']:.4f}",
    f'- 사기유형 최종 모델: {best_type_name}',
    f"- 유형 Macro F1: {type_test['macro_f1']:.4f}",
    f'- 구간 탐지 최종 모델: {best_segment_name}',
    f'- 군집 모델: {best_cluster_key[0]} / k={best_cluster_key[1]}','',
    '## 2-2단계 품질검사 참고','',f'- {quality_note}','',
    '## 해석 시 주의사항','',
    '- 정상상담과 보이스피싱의 출처가 달라 문체·전사 형식 차이를 학습했을 가능성이 있습니다.',
    '- 현재 점수는 우선 두 공개 코퍼스의 구분 성능이며 실서비스 일반화 성능이 아닙니다.',
    '- 사칭·요구행동·심리전략은 규칙 기반 SILVER 라벨입니다.',
    '- 실제 피해 여부 정답이 없어 피해 발생 확률은 예측하지 않았습니다.',
    '- 군집은 잠재 유형 후보이며 정답 분류가 아닙니다.',
]
(REPORT_ROOT/'03_ml_analysis_report.md').write_text('\n'.join(report),encoding='utf-8')

manifest = {
    'seed':SEED,'dataset_root':str(DATASET_ROOT),'output_root':str(OUTPUT_ROOT),
    'best_models':best_models.to_dict(orient='records'),
    'group_leakage_checks_passed':True,
    'saved_models':[path.name for path in MODEL_ROOT.glob('*.joblib')]
}
(REPORT_ROOT/'ml_run_manifest.json').write_text(
    json.dumps(manifest,ensure_ascii=False,indent=2,default=str),encoding='utf-8'
)

assert len(list(MODEL_ROOT.glob('*.joblib'))) == 4
assert det.groupby('group_id')['ml_split'].nunique().max() == 1
assert fraud_type.groupby('group_id')['ml_split'].nunique().max() == 1
assert segment.groupby('group_id')['ml_split'].nunique().max() == 1
print('3단계 정상 완료:',OUTPUT_ROOT)

## 최종 결과 폴더

머신러닝_분석결과_v3 아래에 한글 확인용 데이터, EDA, 데이터 분리표, 모델, 예측 결과, 보고서가 저장됩니다.

금액 원본은 원 단위로 보존하고 분석·시각화에는 만원 단위를 사용합니다. 저장된 joblib은 이 노트북이 만든 신뢰 가능한 파일만 불러오세요.